# 02c · E1 con RigoBERTa Clinical en Google Colab (GPU)

### Por qué este notebook

El notebook 02b entrena RigoBERTa Clinical en el Mac, pero el modelo (~560 millones de parámetros) no cabe bien en su memoria: el sistema pasa a usar el disco y cada época toma más de 2 horas. En una GPU de Colab el mismo entrenamiento toma minutos.

Es **el mismo experimento** que 02b y que E1:
- mismo código (`src/modelos/e1_encoder.py`, clonado desde el repositorio);
- mismos datos, verificados con sus huellas SHA-256;
- mismas 3 semillas y mismos hiperparámetros (lote 16, hasta 5 épocas, *early stopping* con *dev*, lr 2×10⁻⁵).

Lo único que cambia es el hardware. Queda registrado en la huella de cada resultado.

### Cómo usarlo

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU** (T4 sirve; L4 o A100 son más rápidas).
2. **Entorno de ejecución → Ejecutar todas**. Pedirá permiso para conectar Google Drive: ahí se guarda cada resultado apenas termina, así que si Colab se desconecta no se pierde nada. Al volver a correr, retoma donde quedó.
3. Mantener la pestaña abierta. Colab corta las sesiones inactivas.
4. Al final se descarga `rigoberta_resultados.zip`. En el Mac, descomprimirlo **en la carpeta del proyecto**: deja los archivos en `results/e1_completo/` y `results/cache/e1_completo/`. Después correr la sección 3 del notebook 02b (comparación con E1) y el notebook 05 (votación).

In [1]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"
print("GPU:", torch.cuda.get_device_name(0))

name, memory.total [MiB]
Tesla T4, 15360 MiB

GPU: Tesla T4


## 1. Código y datos

Se clona el repositorio del proyecto y el corpus público (PLN@CMM, `acm_health_msen`). Las huellas SHA-256 de `configs/huellas_datos.json` comprueban que los archivos son exactamente los mismos que se usaron en E1.

In [2]:
import os, shutil, glob, subprocess
from pathlib import Path
os.chdir("/content")
def sh(*cmd, cwd=None):
    subprocess.run(cmd, check=True, cwd=cwd)
if not Path("proyecto-ner-aps-genai").exists():
    sh("git", "clone", "-q", "https://github.com/SebasDCIS/proyecto-ner-aps-genai.git")
else:
    sh("git", "pull", "-q", cwd="proyecto-ner-aps-genai")
RAIZ = Path("/content/proyecto-ner-aps-genai")

carpeta = RAIZ / "data" / "raw" / "cwlc"
TIPOS_CORPUS = ["Abbreviation", "Body_Part", "Disease", "Family_Member", "Finding", "Medication", "Procedure"]
if not (carpeta / "Disease" / "Disease_test.conll").exists():
    if not Path("/content/acm_health_msen").exists():
        sh("git", "clone", "-q", "--depth", "1", "https://github.com/plncmm/acm_health_msen.git", "/content/acm_health_msen")
    for tipo in TIPOS_CORPUS:
        (carpeta / tipo).mkdir(parents=True, exist_ok=True)
        for split in ["train", "dev", "test"]:
            origen = glob.glob(f"/content/acm_health_msen/**/{tipo}_{split}.conll", recursive=True)
            assert origen, f"No encontré {tipo}_{split}.conll en el repositorio del corpus"
            shutil.copy(origen[0], carpeta / tipo / f"{tipo}_{split}.conll")
print(sorted(p.name for p in carpeta.iterdir()))

['Abbreviation', 'Body_Part', 'Disease', 'Family_Member', 'Finding', 'Medication', 'Procedure']


In [3]:
import sys, json, time, pickle
sys.path.insert(0, str(RAIZ / "src"))
import yaml
import pandas as pd
from dataclasses import asdict
from datos.cwlc import cargar_split
from modelos.e1_encoder import Config, entrenar, predecir, evaluar, guardar_resultados, dispositivo
from reproducibilidad import huella_completa, verificar_huellas_datos, agregar_semillas

cfg_global = yaml.safe_load(open(RAIZ / "configs" / "experimentos.yaml"))
train, dev, test = (cargar_split(carpeta, s) for s in ["train", "dev", "test"])
ok, difs = verificar_huellas_datos(carpeta, "cwlc")
assert ok, f"¡Los datos no coinciden con los de E1! {difs}"
print(f"train={len(train)} dev={len(dev)} test={len(test)} | dispositivo: {dispositivo()} | huellas verificadas")

train=8014 dev=890 test=990 | dispositivo: cuda | huellas verificadas


## 2. Guardar en Google Drive

Cada resultado se escribe en `Mi unidad/proyecto-ner-aps-colab/` apenas termina.

In [4]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/proyecto-ner-aps-colab")
SALIDA = DRIVE / "results" / "e1_completo"
CACHE = DRIVE / "results" / "cache" / "e1_completo"
SALIDA.mkdir(parents=True, exist_ok=True); CACHE.mkdir(parents=True, exist_ok=True)
print(SALIDA)

Mounted at /content/drive
/content/drive/MyDrive/proyecto-ner-aps-colab/results/e1_completo


## 3. Configuración

Igual que E1: **lote 16 sin acumulación**. Si aparece `CUDA out of memory` (posible en una T4 con oraciones largas), cambiar a `BATCH, ACUMULAR = 8, 2`: el lote efectivo sigue siendo 16.

In [5]:
CLAVE = "rigoberta_clinico"
NOMBRE = cfg_global["encoders"][CLAVE]                  # IIC/RigoBERTa-Clinical
g = cfg_global["entrenamiento_encoder_grande"]
BATCH, ACUMULAR = 8, 2
TIPOS = cfg_global["entidades"]
SEMILLAS = cfg_global["semillas"]

def config_para(tipo, semilla):
    return Config(modelo=NOMBRE, tipo=tipo, epocas=g["epocas"], lr=g["lr"], batch=BATCH, acumular=ACUMULAR,
                  max_len=g["max_len"], semilla=semilla)

pendientes = [(s, t) for s in SEMILLAS for t in TIPOS if not (SALIDA / f"{CLAVE}_{t}_s{s}.json").exists()]
print(NOMBRE, "| lr", g["lr"], "| lote", BATCH, "× acumulación", ACUMULAR, f"| pendientes: {len(pendientes)} de {len(SEMILLAS) * len(TIPOS)}")

IIC/RigoBERTa-Clinical | lr 2e-05 | lote 16 × acumulación 1 | pendientes: 9 de 9


## 4. Entrenar

Orden: semilla por semilla. Durante cada época se imprime la velocidad (segundos por lote y minutos restantes).

In [6]:
for semilla in SEMILLAS:
    for tipo in TIPOS:
        ruta = SALIDA / f"{CLAVE}_{tipo}_s{semilla}.json"
        ruta_pred = CACHE / f"{CLAVE}_{tipo}_s{semilla}.pkl"
        if ruta.exists() and ruta_pred.exists():
            print(f"(ya existe) {ruta.name}"); continue
        print(f"\n=== {CLAVE} · {tipo} · semilla {semilla} · {time.strftime('%H:%M')} UTC ===")
        cfg = config_para(tipo, semilla)
        t0 = time.time()
        modelo, cod, historial = entrenar(cfg, train, dev)
        pred = predecir(modelo, cod, test, tipo, disp=dispositivo())
        gold = [e for o in test for e in o.entidades if e[3] == tipo]
        res = evaluar(gold, pred, test, tipo, train=train)
        res["clave"], res["minutos_total"] = CLAVE, round((time.time() - t0) / 60, 1)
        res["hardware"] = f"Colab · {torch.cuda.get_device_name(0)}"
        huella = huella_completa(asdict(cfg), carpeta, modelo=NOMBRE, splits={"train": train, "dev": dev, "test": test})
        ruta_pred.write_bytes(pickle.dumps(pred))
        guardar_resultados(ruta, cfg, historial, res, huella)
        print(f"  F1 estricto = {res['estricto']['f1']:.3f} · {res['minutos_total']} min")
        del modelo; torch.cuda.empty_cache()


=== rigoberta_clinico · Disease · semilla 42 · 14:18 UTC ===


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

... (4 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: IIC/RigoBERTa-Clinical
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/at

    lote 25/501 · 1.0 s por lote · faltan ~8 min de esta época


OutOfMemoryError: CUDA out of memory. Tried to allocate 90.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 55.81 MiB is free. Including non-PyTorch memory, this process has 14.51 GiB memory in use. Of the allocated memory 14.14 GiB is allocated by PyTorch, and 186.35 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 5. Resultados de RigoBERTa

Para comparar con E1, la referencia (encoder español clínico / chileno clínico, media de 3 semillas, F1 estricto) es: Disease 0,798 / 0,815 · Medication 0,840 / 0,794 · Abbreviation 0,950 / 0,952. La comparación pareada por semilla se hace en el Mac (notebook 02b, sección 3).

In [ ]:
filas = []
for f in sorted(SALIDA.glob(f"{CLAVE}_*_s*.json")):
    d = json.loads(f.read_text())
    filas.append({"tipo": d["config"]["tipo"], "semilla": d["config"]["semilla"],
                  "F1 estricto": d["resultados"]["estricto"]["f1"], "F1 laxo": d["resultados"]["laxo"]["f1"],
                  "épocas": len([h for h in d["historial"] if "epoca" in h]) if "historial" in d else None,
                  "min": d["resultados"].get("minutos_total")})
tab = pd.DataFrame(filas)
display(tab)
def media_de(x):
    a = agregar_semillas(list(x)); return f"{a['media']:.3f} ± {a['de']:.3f} (n={a['n']})"
tab.groupby("tipo")[["F1 estricto", "F1 laxo"]].agg(media_de)

## 6. Descargar para el Mac

Descomprimir `rigoberta_resultados.zip` en la carpeta raíz del proyecto en el Mac.

In [ ]:
import zipfile
from google.colab import files
zip_ruta = Path("/content/rigoberta_resultados.zip")
with zipfile.ZipFile(zip_ruta, "w") as z:
    for f in SALIDA.glob(f"{CLAVE}_*"):
        z.write(f, f"results/e1_completo/{f.name}")
    for f in CACHE.glob(f"{CLAVE}_*"):
        z.write(f, f"results/cache/e1_completo/{f.name}")
print(zipfile.ZipFile(zip_ruta).namelist())
files.download(str(zip_ruta))